### 🔰PyTorchでニューラルネットワーク基礎 #33 Dataset編

### 内容
* Qiitaの記事と連動しています
* 各種ファイルの保存先は環境によって適宜変更してください


### データについて
* ramen_data.csv: 架空のデータ（表タイプ、分類問題）
* history_text_label_id.jsonl: wikipediaの日本の歴史分野の内容から収集した時代区分を判定するテキスト分類問題
    * 時代区分を直接表すテキストは除いてある。（江戸時代ラベルの文章に「江戸」という単語がはありません）
    * クラス数：５（弥生、奈良、室町、江戸、昭和）
* greek_wiki.txt: wikipediaのギリシャ神話から収集したテキストデータ


### トークナイザーについて
* unigram_tokenizer_2k.json：第31回の事前学習で利用したトークナイザー。unigram lmと日本の歴史分野のコーパスで学習した2000トークン。
* greek_unigram_tokenizer_3k: wikipediaのギリシア神話テキストを利用して学習したトークナイザーで3000トークン。

### Datasetクラスのカスタマイズ
ライブラリーの読み込み

In [ ]:
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset
# 等長化・paddingに利用
from torch.nn.utils.rnn import pad_sequence

# tokenizer使う時に利用
from tokenizers import Tokenizer
from tokenizers.processors import TemplateProcessing

### 基本形

In [ ]:
class MyDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        return self.data[index]

In [ ]:
data = [3, 2, 1, 0, 1, 2, 3]
dataset = MyDataset(data)

# (1) __len__の使い方
print(dataset.__len__())   # 7
print(len(dataset))        # 7　通常の書き方

# (2) __getitem__の使い方 
print(dataset.__getitem__(1))  # 2
print(dataset[1]) 

## 1.2 Numpy配列用に修正したDatasetクラス

In [ ]:
datafile = "./data/ramen_data.csv"
x = np.loadtxt(datafile, delimiter=",", skiprows=1, usecols=(0,1,2,3,4))
t = np.loadtxt(datafile, delimiter=",", skiprows=1, usecols=(6))

In [ ]:
class MyDataset02(Dataset):
    # (1) 引数が２種類(x, t)
    def __init__(self, x,t):
        self.x = x   # 入力データ
        self.t = t   # 教師データ
   
   # (2) どちらかのサイズでOK
    def __len__(self):
        return len(self.t)
    
    # (3) 入出力するデータのタイプによって適宜修正
    def __getitem__(self, index):
        return {
            "data":  np.array(self.x[index], dtype=np.float32),
            "label": np.array(self.t[index], dtype=np.int64)
            }


In [ ]:
dataset = MyDataset02(x,t)

print(len(x))   # 500
print(len(t))   # 500
print(dataset[0])          # {'data': array([3.2, 3.9, 5.5, 1.9, 8.7], dtype=float32), 'label': array(0)}
print(dataset[0]["data"])  # [3.2 3.9 5.5 1.9 8.7]
print(dataset[0]["label"]) # 0

**torch.utils.data.random_split**
* Datasetオブジェクトをランダムに分割することができる

In [ ]:
from torch.utils.data import random_split
train, test = random_split(dataset, [0.8, 0.2])
len(train), len(test)

## 1.3 データフレーム用に修正したDatasetクラス

In [ ]:
data_filename = "./data/history_text_label_id.jsonl"     # 分類問題のデータ
df = pd.read_json(data_filename, lines=True)

In [ ]:
df.sample(3)

In [ ]:
class MyDataset03(Dataset):
    # (1) dataがデータフレームになっている
    def __init__(self, data):
        self.data = data
   
    def __len__(self):
        return len(self.data)
    
    # (2) データの形式で適宜修正
    def __getitem__(self, index):
        item = self.data.iloc[index]   # データフレームのindex行を取得したいので data.iloc[]を使う
        tensor_x = torch.tensor(item["ids"], dtype=torch.long)
        tensor_t = torch.tensor(item["label"], dtype=torch.long)
        return {"ids": tensor_x, "label": tensor_t}
        # 次の形だとdfの値なのでリストや数値でreturnとなります
        #return {"ids": item["ids"], "label": item["label"]}

In [ ]:
dataset = MyDataset03(df)
print(len(dataset)) # 1000
print(dataset[1])

## 1.4 データフレーム版・\_\_init\_\_でリストにするパターン

In [ ]:
class MyDataset04(Dataset):
    # (1) データフレームをtolist()でリストへ
    def __init__(self, data):
        self.ids = data["ids"].tolist()
        self.label = data["label"].tolist()
   
    def __len__(self):
        return len(self.label)
    
    # (2) (1)でリストになっているので添字アクセス可能
    def __getitem__(self, index):
        return {"ids":   self.ids[index],
                "label": self.label[index]}

In [ ]:
dataset = MyDataset04(df)
print(len(dataset))   # 1000
print(dataset[1])

# 応用編
## 2.1 前処理も入れてしまうパターン (\_\_init\_\_部分)

In [ ]:
class MyDataset05(Dataset):
    # pad_sequenceを追加して、すべてのidsが等長化させるぞ〜
    def __init__(self, data):
        # (1) 各idsをtorch.tensorに変換、リストとします。
        ids_list = [torch.tensor(x, dtype=torch.long) for x in data["ids"]]
        labels = torch.tensor(data["label"].tolist(), dtype=torch.long)
        # (2) dataのids列をすべて等長化
        ids_padded = pad_sequence(ids_list, batch_first=True, padding_value=0)
        self.ids_padded = ids_padded
        self.labels = labels
   
    def __len__(self):
        return len(self.labels)
    
    # (3) ids_paddedやlabelsはテンソルになっています
    def __getitem__(self, index):
        return {"ids": self.ids_padded[index], 
                "label": self.labels[index]
                }

In [ ]:
dataset = MyDataset05(df)
dataset[0]

## 2.2 前処理も入れてしまうパターン２（\_\_getitem\_\_部分）

In [ ]:
class MyDataset06(Dataset):
    # (1) 準備
    def __init__(self, data, max_length=64):
        self.ids = data["ids"].tolist()        # paddingしやすいようにリスト化
        self.label = data["label"].tolist()
        self.max_length = max_length           # 最大系列長を指定

    def __len__(self):
        return len(self.label)
        
    def __getitem__(self, index):
        # (2) index行をmax_lengthまでpadding
        ids = self.ids[index]                  # index番目のindex行だけ取り出す
        ids = ids[:self.max_length]            # 長すぎたら切り詰める
        ids = ids + [0] * (self.max_length - len(ids))  # 短い分を0で埋める
        return {
            "ids": torch.tensor(ids, dtype=torch.long),
            "label": torch.tensor(self.label[index], dtype=torch.long),
        }

In [ ]:
dataset = MyDataset06(df)
dataset[0]

DataLoaderを利用して、等長化されているのか確認してみた

In [ ]:
from torch.utils.data import DataLoader
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)
next(iter(dataloader))

## 2.3 tokenizerを\_\_init\_\_に入れてみた

In [ ]:
# (1) idsで利用したtokenizerと同じものを指定
tokenizer_filename = "tokenizer/unigram_tokenizer_2k.json"
tokenizer = Tokenizer.from_file(tokenizer_filename)

bos_id = tokenizer.token_to_id("<bos>")
eos_id = tokenizer.token_to_id("<eos>")

# (2) encodeするときのテンプレ
tokenizer.post_processor = TemplateProcessing(
    single="<bos> $A <eos>",
    special_tokens=[("<bos>", bos_id), ("<eos>", eos_id)],
)


class Dataset07(Dataset):
    def __init__(self, data, tokenizer):
        # (3) encode_batchで一度にid化
        # self.tokenizer = tokenizer Dataset08では有効化
        encodings = tokenizer.encode_batch(list(data["text"]))
        ids_list = [torch.tensor(e.ids, dtype=torch.long) for e in encodings]

        # (4) 確認用　ids列の値と比較する
        org_ids_list = [torch.tensor(x, dtype=torch.long) for x in data["ids"]]
        # (5) ラベル
        labels = torch.tensor(data["label"].tolist(), dtype=torch.long)        
        # (6) 一応、等長化してみた
        pad_id = tokenizer.token_to_id("<pad>")
        ids_padded = pad_sequence(ids_list, batch_first=True, padding_value=pad_id)
        self.ids_padded = ids_padded
        self.labels = labels
        self.org_ids_list = org_ids_list
   
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, index):
        return {"ids": self.ids_padded[index],
                "org_ids": self.org_ids_list[index],
                "label": self.labels[index]
                }

In [ ]:
dataset = Dataset07(df, tokenizer)
dataset[0]

## 2.4 tokenizerを\_\_getitem\_\_に入れてみた

In [ ]:
class Dataset08(Dataset):
    def __init__(self, data, tokenizer):
        self.tokenizer = tokenizer # getitemで使う
        # トークン化はせず、textはそのまま保持しておく
        # (1)
        self.text_list = data["text"].tolist()
        self.org_ids_list = [torch.tensor(x, dtype=torch.long) for x in data["ids"]]
        self.labels = torch.tensor(data["label"].tolist(), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        # (2) 取り出すタイミングで1件だけencodeする
        ids = torch.tensor(self.tokenizer.encode(self.text_list[index]).ids, dtype=torch.long)
        
        return {"ids": ids,
                "org_ids": self.org_ids_list[index],
                "label": self.labels[index]}

In [ ]:
dataset = Dataset08(df, tokenizer)
dataset[0]

## 2.5 テキストデータ（Transformer Decoderタイプ）

In [ ]:
tokenizer_filename = "tokenizer/greek_unigram_tokenizer_3k.json"
tokenizer = Tokenizer.from_file(tokenizer_filename)

class MyDataset09(Dataset):
    # (1) tokenizer.encode(text).idsがtextのID列
    def __init__(self, text, tokenizer, context_size=16, stride=2):
        self.context_size = context_size
        self.ids = torch.tensor(tokenizer.encode(text).ids, dtype=torch.long)
        # 各サンプルの開始位置をあらかじめ計算
        self.starts = list(range(0, len(self.ids) - context_size, stride))
    # データ数
    def __len__(self):
        return len(self.starts)

    # (2)
    # index番号からcontext_sizeまでが入力データ
    # index番号＋１からcontext_sizeまでが次のトークン予測のラベルデータ
    def __getitem__(self, idx):
        # 入力とターゲットのペアを作成
        start = self.starts[idx]
        x = self.ids[start:start + self.context_size]
        y = self.ids[start + 1:start + self.context_size + 1]
        return {"ids": x, "labels": y}

with open("./data/greek_wiki.txt","rt") as file:
    text = file.read()

dataset = MyDataset09(text, tokenizer, context_size=16, stride=2)

print(f"{dataset[0]=}")
print(f"{dataset[1]=}")